# Comprendre une fonction complexe

In [ ]:
class Seq2SeqTransformer(nn.Module):
    def __init__(
        self,
        num_encoder_layers: int,
        num_decoder_layers: int,
        emb_size: int,
        nhead: int,
        src_vocab_size: int,
        tgt_vocab_size: int,
        dim_feedforward: int = 512,
        dropout: float = 0.1
    ):
        super(Seq2SeqTransformer, self).__init__()
        self.transformer = Transformer(
            d_model=emb_size,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.generator = nn.Linear(emb_size, tgt_vocab_size)
        self.src_tok_emb = TokenEmbedding(src_vocab_size, emb_size)
        self.tgt_tok_emb = TokenEmbedding(tgt_vocab_size, emb_size)
        self.positional_encoding = PositionalEncoding(
            emb_size, dropout=dropout)

    def forward(self,
                src: Tensor,
                trg: Tensor,
                src_mask: Tensor,
                tgt_mask: Tensor,
                src_padding_mask: Tensor,
                tgt_padding_mask: Tensor,
                memory_key_padding_mask: Tensor):
        src_emb = self.positional_encoding(self.src_tok_emb(src))
        tgt_emb = self.positional_encoding(self.tgt_tok_emb(trg))
        outs = self.transformer(src_emb, tgt_emb, src_mask, tgt_mask, None,
                                src_padding_mask, tgt_padding_mask, memory_key_padding_mask)
        return self.generator(outs)

    def encode(self, src: Tensor, src_mask: Tensor):
        return self.transformer.encoder(self.positional_encoding(
                            self.src_tok_emb(src)), src_mask)

    def decode(self, tgt: Tensor, memory: Tensor, tgt_mask: Tensor):
        return self.transformer.decoder(self.positional_encoding(
                          self.tgt_tok_emb(tgt)), memory,
                          tgt_mask)

# Créer une docstring

In [1]:
def split_sentence_to_words(sentence: str) -> list:

    return sentence.split()

# Créer une fonction de test

# Le refactoring

Pour chaque élément x dans data, la fonction :
- Multiplie par 2 si x est pair, ou par 3 si x est impair.
- Ensuite, elle applique une modification en fonction de si la valeur résultante est supérieure à 10 ou non.

In [ ]:
def process_data(data):
    result = []
    for i in range(len(data)):
        temp = data[i]
        if temp % 2 == 0:
            temp = temp * 2
        else:
            temp = temp * 3
        result.append(temp)
    for i in range(len(result)):
        if result[i] > 10:
            result[i] = result[i] - 1
        else:
            result[i] = result[i] + 1
    return result

**Redondance** : Le code utilise des boucles for séparées pour deux transformations différentes sur les données, ce qui peut être combiné pour plus d'efficacité.

**Lisibilité** : La logique de transformation des données est difficile à suivre en raison de l'utilisation de variables temporaires et de plusieurs boucles.

**Optimisation** : Le code peut être optimisé en combinant les étapes et en utilisant des constructions Python plus idiomatiques comme les compréhensions de liste.